<a href="https://colab.research.google.com/github/Gagaririna/Ya_practicum/blob/main/project45_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Сборный проект-4
# Поиск изображений по текстовому запросу — Proof of Concept


Проект для фотохостинга **«Со Смыслом» (With Sense)**: построить демонстрационную модель, которая оценивает соответствие текста и изображения числом от 0 до 1 и показывает наиболее релевантные изображения по пользовательскому описанию.

### Описание данных

Данные доступны по [ссылке](https://code.s3.yandex.net/datasets/dsplus_integrated_project_4.zip).

В файле `train_dataset.csv` находится информация, необходимая для обучения: имя файла изображения, идентификатор описания и текст описания. Для одной картинки может быть доступно до 5 описаний. Идентификатор описания имеет формат `<имя файла изображения>#<порядковый номер описания>`.

В папке `train_images` содержатся изображения для тренировки модели.

В файле `CrowdAnnotations.tsv` — данные по соответствию изображения и описания, полученные с помощью краудсорсинга. Номера колонок и соответствующий тип данных:

1. Имя файла изображения.
2. Идентификатор описания.
3. Доля людей, подтвердивших, что описание соответствует изображению.
4. Количество человек, подтвердивших, что описание соответствует изображению.
5. Количество человек, подтвердивших, что описание не соответствует изображению.

В файле `ExpertAnnotations.tsv` содержатся данные по соответствию изображения и описания, полученные в результате опроса экспертов. Номера колонок и соответствующий тип данных:

1. Имя файла изображения.
2. Идентификатор описания.

3, 4, 5 — оценки трёх экспертов.

Эксперты ставят оценки по шкале от 1 до 4, где 1 — изображение и запрос совершенно не соответствуют друг другу, 2 — запрос содержит элементы описания изображения, но в целом запрос тексту не соответствует, 3 — запрос и текст соответствуют с точностью до некоторых деталей, 4 — запрос и текст соответствуют полностью.

В файле `test_queries.csv` находится информация, необходимая для тестирования: идентификатор запроса, текст запроса и релевантное изображение. Для одной картинки может быть доступно до 5 описаний. Идентификатор описания имеет формат `<имя файла изображения>#<порядковый номер описания>`.

В папке `test_images` содержатся изображения для тестирования модели.

## 2. Проверка данных

В некоторых странах, где работает ваша компания, действуют ограничения по обработке изображений: поисковым сервисам и сервисам, предоставляющим возможность поиска, запрещено без разрешения родителей или законных представителей предоставлять любую информацию, в том числе, но не исключительно тексты, изображения, видео и аудио, содержащие описание, изображение или запись голоса детей. Ребёнком считается любой человек, не достигший 16 лет.

В вашем сервисе строго следуют законам стран, в которых работают. Поэтому при попытке посмотреть изображения, запрещённые законодательством, вместо картинок показывается дисклеймер:

> This image is unavailable in your country in compliance with local laws
>

Однако у вас в PoC нет возможности воспользоваться данным функционалом. Поэтому все изображения, которые нарушают данный закон, нужно удалить из обучающей выборки.

## Загрузка данных

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
!pip install -U sentence-transformers --quiet

In [ ]:
!pip install "pillow<12" -q

In [ ]:
!pip install nltk -q

In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
import json
import os
from functools import lru_cache
from json import JSONDecodeError
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import re
from functools import lru_cache
from typing import Iterable
import pandas as pd
from pathlib import Path
from typing import Iterable

import seaborn as sns

from typing import Iterable

import numpy as np
import pandas as pd

import numpy as np
import pandas as pd

import matplotlib.font_manager as fm
import warnings
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet
from collections import Counter
from wordcloud import WordCloud
from random import sample
from sentence_transformers import SentenceTransformer
import torch
import torchvision

Для быстрой проверки можно задать переменную окружения DEV_SAMPLE_LIMIT, например DEV_SAMPLE_LIMIT=80.

In [ ]:
nltk.download('stopwords') #Скачаем набор "стоп-слов", которые обычно не несут смысловой нагрузки для анализа текста и часто удаляются на этапе предобработки.
nltk.download('averaged_perceptron_tagger') #скачаем одель для части речи — POS-теггера (Part-Of-Speech Tagger)
stop_words = set(stopwords.words('english')) #устанавливаем сет стоп-слов на англ языке (датасет представлен на английском)
lemmatizer = WordNetLemmatizer() #Создается объект лемматизатора
nltk.download('all')

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
path = './dsplus_integrated_project_4/to_upload/'

In [ ]:
train_image_dir = path+"train_images"
test_image_dir = path+"test_images"

In [ ]:
train_image_dir = Path(train_image_dir)
test_image_dir = Path(test_image_dir)

In [ ]:
fonts = fm.findSystemFonts(fontext='ttf')

In [ ]:
disclaimer = "This image is unavailable in your country in compliance with local laws."

In [ ]:
ban_words: tuple[str, ...] = (
    "adolescent",
    "adolescents",
    "baby",
    "babies",
    "bairn",
    "bambino",
    "boy",
    "boys",
    "child",
    "children",
    "daughter",
    "girl",
    "girls",
    "infant",
    "juvenile",
    "kid",
    "kids",
    "kiddie",
    "kiddy",
    "kiddies",
    "minor",
    "minors",
    "neonate",
    "newborn",
    "offspring",
    "preteen",
    "preschooler",
    "pupil",
    "pupils",
    "schoolchild",
    "son",
    "teen",
    "teens",
    "teenage",
    "teenager",
    "teenagers",
    "toddler",
    "toddlers",
    "tot",
    "tots",
    "tween",
    "under 16",
    "under-16",
    "under-16s",
    "young person",
    "youngster",
    "youngsters",
    "youth",
    "youths",
)

In [ ]:
def info_data(df):
    df.info()
    display(df.describe())
    print("\nПроверка на наличие пропусков:")
    null_values = df.isnull().sum()
    if null_values.sum() > 0:
            print("В датафрейме есть пропущенные значения:")
            print(null_values)
    else:
        print("Пропущенные значения в датафрейме отсутствуют.")

    print("\nПроверка на наличие дубликатов:")
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"В датафрейме найдено {duplicates} явных дубликатов.")
    else:
        print("Явные дубликаты в датафрейме отсутствуют.")

    display(df.head())

In [ ]:
def apply_legal_filter_frame(frame, apply_legal_filter=True):
    """
    Фильтрует датасет от restricted текстов и связанных изображений
    и возвращает очищенный frame + статистику.
    """

    input_rows = len(frame)

    if apply_legal_filter:

        # 1. помечаем restricted тексты
        bad_text = frame["query_text"].map(contains_restricted_content)

        # 2. изображения, связанные с плохими текстами и значимым score
        bad_images = set(
            frame.loc[
                bad_text & frame["final_score"].fillna(0).gt(0),
                "image"
            ]
        )

        # 3. фильтрация
        frame_clean = frame.loc[
            ~bad_text & ~frame["image"].isin(bad_images)
        ].copy()

        stats = {
            "input_rows": input_rows,
            "restricted_text_rows": int(bad_text.sum()),
            "restricted_relevant_images": len(bad_images),
            "output_rows": len(frame_clean),
        }

    else:

        frame_clean = frame.copy()

        stats = {
            "input_rows": input_rows,
            "restricted_text_rows": 0,
            "restricted_relevant_images": 0,
            "output_rows": len(frame_clean),
        }

    return frame_clean.reset_index(drop=True), stats

In [ ]:
def _term_to_pattern(term: str) -> str:
    """Convert a term into a regex fragment with flexible space/hyphen matching."""
    escaped = re.escape(term).replace(r"\ ", r"[\s-]+")
    return escaped

In [ ]:
@lru_cache(maxsize=1)
def blocked_terms_pattern() -> re.Pattern[str]:
    """Return compiled case-insensitive regex for blocked terms.

    Boundaries avoid matching terms inside larger words (e.g. ``boy`` inside
    ``boycott``). Multi-word terms allow whitespace or hyphen separators.
    """
    alternatives = sorted((_term_to_pattern(t) for t in ban_words), key=len, reverse=True)
    pattern = r"(?<![A-Za-z])(" + "|".join(alternatives) + r")(?![A-Za-z])"
    return re.compile(pattern, flags=re.IGNORECASE)


In [ ]:
def restricted_terms_in_text(text: object) -> list[str]:
    """Return unique restricted terms found in ``text`` in first-seen order."""
    if text is None:
        return []
    normalized = str(text)
    seen: set[str] = set()
    terms: list[str] = []
    for match in blocked_terms_pattern().finditer(normalized):
        value = match.group(0).lower()
        value = re.sub(r"[\s-]+", " ", value).strip()
        if value not in seen:
            seen.add(value)
            terms.append(value)
    return terms


In [ ]:
def contains_restricted_content(text: object) -> bool:
    """Return True when ``text`` contains child/minor-related blocked terms."""
    return bool(restricted_terms_in_text(text))


In [ ]:
def filter_safe_texts(texts: Iterable[object]) -> list[bool]:
    """Return a boolean mask where True means the corresponding text is safe."""
    return [not contains_restricted_content(text) for text in texts]

In [ ]:
def info_data(df):
    df.info()
    display(df.describe())
    print("\nПроверка на наличие пропусков:")
    null_values = df.isnull().sum()
    if null_values.sum() > 0:
            print("В датафрейме есть пропущенные значения:")
            print(null_values)
    else:
        print("Пропущенные значения в датафрейме отсутствуют.")

    print("\nПроверка на наличие дубликатов:")
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"В датафрейме найдено {duplicates} явных дубликатов.")
    else:
        print("Явные дубликаты в датафрейме отсутствуют.")

    display(df.head())

### функции: подготовка таблиц

In [ ]:
def limit_by_rows(frame, limit, random_state=42):
    if limit <= 0 or len(frame) <= limit:
        return frame.reset_index(drop=True)
    return frame.sample(n=limit, random_state=random_state).reset_index(drop=True)


def missing_image_files(image_dir, image_names):
    return [name for name in sorted(set(image_names)) if not (Path(image_dir) / name).exists()]


def build_safe_test_frames(test_queries, test_images, image_col="image", query_col="query_text"):
    # Для честного демо оставляем только безопасную часть официального test split.
    queries = test_queries.copy()
    images = test_images.copy()

    restricted_query = queries[query_col].map(contains_restricted_content)
    restricted_images = set(queries.loc[restricted_query, image_col])

    safe_images = images.loc[~images[image_col].isin(restricted_images)].copy()
    safe_queries = queries.loc[
        ~restricted_query & queries[image_col].isin(set(safe_images[image_col]))
    ].copy()

    stats = {
        "input_queries": int(len(queries)),
        "restricted_query_rows": int(restricted_query.sum()),
        "input_images": int(images[image_col].nunique()),
        "restricted_images": int(len(restricted_images)),
        "safe_queries": int(len(safe_queries)),
        "safe_images": int(safe_images[image_col].nunique()),
    }
    return safe_queries.reset_index(drop=True), safe_images.reset_index(drop=True), stats


### функции: признаки текста и картинок

In [ ]:


SENTENCE_TRANSFORMER_DEFAULT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


def l2_normalize_rows(matrix, eps=1e-12):
    arr = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1, keepdims=True).astype(np.float32)
    norms = np.where(norms < eps, 1.0, norms)
    return (arr / norms).astype(np.float32)


@lru_cache(maxsize=4)
def load_sentence_transformer(model_name, device=None):

    target_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    return SentenceTransformer(model_name, device=target_device)


def compute_sentence_transformer_text_features(texts, model_name=SENTENCE_TRANSFORMER_DEFAULT_MODEL, batch_size=64, device=None):
    model = load_sentence_transformer(model_name, device=device)
    embeddings = model.encode(
        [str(t) for t in texts],
        batch_size=batch_size,
        show_progress_bar=os.getenv("WITHSENSE_PROGRESS", "0") == "1",
        normalize_embeddings=False,
        convert_to_numpy=True,
    ).astype(np.float32)
    return model, embeddings


def encode_text(model, texts, batch_size=64):
    return model.encode(
        [str(t) for t in texts],
        batch_size=batch_size,
        show_progress_bar=False,
        normalize_embeddings=False,
        convert_to_numpy=True,
    ).astype(np.float32)


def load_resnet50(device=None):
    import torch
    from torchvision import models

    target_device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights).to(target_device).eval()
    preprocess = weights.transforms()
    feature_extractor = torch.nn.Sequential(*(list(model.children())[:-1])).to(target_device).eval()
    return torch, model, feature_extractor, preprocess, target_device, "imagenet_pretrained"


def load_npz_cache(cache_path, expected_names):
    if cache_path is None or not Path(cache_path).exists():
        return None
    data = np.load(cache_path, allow_pickle=False)
    names = [str(x) for x in data["names"].tolist()]
    embeddings = data["embeddings"].astype(np.float32)
    if not set(expected_names).issubset(names):
        return None
    name_to_idx = {name: i for i, name in enumerate(names)}
    order = [name_to_idx[name] for name in expected_names]
    return expected_names, embeddings[order]


def compute_resnet50_embeddings(image_dir, image_names, cache_path=None, batch_size=32, device=None, include_logits=True):
    unique_names = list(dict.fromkeys(str(name) for name in image_names))
    if not unique_names:
        raise ValueError("No image names provided")

    if cache_path is not None:
        cache_path = Path(cache_path)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cached = load_npz_cache(cache_path, unique_names)
        if cached is not None:
            names, embeddings = cached
            meta_path = cache_path.with_suffix(".json")
            status = "loaded_from_cache"
            if meta_path.exists():
                try:
                    status = json.loads(meta_path.read_text()).get("weights_status", status)
                except (OSError, JSONDecodeError, TypeError, AttributeError) as exc:
                    status = f"loaded_from_cache_metadata_unavailable:{exc.__class__.__name__}"
            return names, embeddings, status

    torch, full_model, feature_extractor, preprocess, target_device, status = load_resnet50(device=device)
    vectors = []
    image_dir = Path(image_dir)

    for start in tqdm(range(0, len(unique_names), batch_size), desc="ResNet50 embeddings"):
        batch_names = unique_names[start : start + batch_size]
        tensors = []
        for name in batch_names:
            path = image_dir / name
            if not path.exists():
                raise FileNotFoundError(path)
            with Image.open(path) as image:
                tensors.append(preprocess(image.convert("RGB")))

        batch = torch.stack(tensors).to(target_device)
        with torch.no_grad():
            pooled = feature_extractor(batch).flatten(1)
            if include_logits:
                logits = full_model(batch)
                features = torch.cat([pooled, logits], dim=1)
            else:
                features = pooled
        vectors.append(features.detach().cpu().numpy().astype(np.float32))

    embeddings = np.vstack(vectors).astype(np.float32)
    if cache_path is not None:
        np.savez_compressed(cache_path, names=np.array(unique_names), embeddings=embeddings)
        cache_path.with_suffix(".json").write_text(json.dumps({"weights_status": status, "include_logits": include_logits, "dim": int(embeddings.shape[1])}, indent=2))
    return unique_names, embeddings, status


def image_feature_matrix(row_image_names, embedding_names, embedding_matrix):
    index = {name: i for i, name in enumerate(embedding_names)}
    missing = sorted({str(name) for name in row_image_names if str(name) not in index})
    if missing:
        raise KeyError(f"Missing embeddings for images: {missing[:5]}")
    return np.vstack([embedding_matrix[index[str(name)]] for name in row_image_names]).astype(np.float32)


### Небольшие функции: модели и метрики

Без пользовательских классов: sklearn-модели хранятся как словари, а нейросеть собрана из стандартных слоёв PyTorch.

In [ ]:



def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)) if len(np.unique(y_true)) > 1 else float("nan"),
    }


def grouped_train_val_indices(groups, train_size=0.7, random_state=42):
    dummy_y = np.zeros(len(groups), dtype=np.float32)
    dummy_x = np.zeros((len(groups), 1), dtype=np.float32)
    splitter = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    return next(splitter.split(dummy_x, dummy_y, groups=groups))


def check_no_group_overlap(groups, train_idx, val_idx):
    groups = np.asarray(groups)
    overlap = set(groups[train_idx]).intersection(set(groups[val_idx]))
    if overlap:
        raise AssertionError(f"Group leakage detected: {sorted(overlap)[:5]}")


def make_torch_interaction_model(image_dim, text_dim, hidden=384, dropout=0.15, device=None):
    import torch
    import torch.nn as nn

    target_device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    image_proj = nn.Linear(image_dim, text_dim).to(target_device)
    mlp = nn.Sequential(
        nn.Linear(4 * text_dim, hidden),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, hidden // 2),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden // 2, 1),
        nn.Sigmoid(),
    ).to(target_device)
    return {
        "name": "Fully connected neural network",
        "kind": "torch_interaction",
        "image_proj": image_proj,
        "mlp": mlp,
        "image_dim": image_dim,
        "text_dim": text_dim,
        "device": str(target_device),
    }


def torch_forward(model, x):
    import torch
    import torch.nn.functional as F

    image_dim = model["image_dim"]
    img = x[:, :image_dim]
    txt = x[:, image_dim:]
    img_proj = model["image_proj"](img)
    img_proj = F.normalize(img_proj, p=2, dim=1, eps=1e-12)
    txt = F.normalize(txt, p=2, dim=1, eps=1e-12)
    features = torch.cat([img_proj, txt, img_proj * txt, (img_proj - txt).abs()], dim=1)
    return model["mlp"](features)


def predict_model(model, X, clip=True):
    X = np.asarray(X, dtype=np.float32)
    if model["kind"] == "sklearn":
        pred = np.asarray(model["estimator"].predict(X)).reshape(-1)
    else:
        import torch

        model["image_proj"].eval()
        model["mlp"].eval()
        device = model["device"]
        with torch.no_grad():
            tensor = torch.as_tensor(X, dtype=torch.float32, device=device)
            pred = torch_forward(model, tensor).detach().cpu().numpy().reshape(-1)
    if clip:
        pred = np.clip(pred, 0.0, 1.0)
    return pred.astype(np.float32)


def train_torch_fnn(X_train, y_train, X_val, y_val, image_dim, text_dim, random_state=42, epochs=200, batch_size=128, learning_rate=1e-3, weight_decay=1e-4, hidden=384, dropout=0.15, loss_name="weighted_mse"):
    import torch
    import torch.nn as nn

    torch.manual_seed(random_state)
    model = make_torch_interaction_model(image_dim, text_dim, hidden=hidden, dropout=dropout)
    params = list(model["image_proj"].parameters()) + list(model["mlp"].parameters())
    optimizer = torch.optim.AdamW(params, lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    device = model["device"]

    X_train_t = torch.as_tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.as_tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
    X_val_t = torch.as_tensor(X_val, dtype=torch.float32, device=device)
    y_val_t = torch.as_tensor(y_val.reshape(-1, 1), dtype=torch.float32, device=device)

    weights = torch.ones_like(y_train_t)
    weights = weights + (y_train_t > 0).float() * 4.0
    weights = weights + (y_train_t >= 0.5).float() * 3.0

    best_state = None
    best_val = float("inf")
    patience = 30
    stale = 0
    n = len(X_train)
    batch_size = max(1, min(batch_size, n))

    for epoch in range(max(1, epochs)):
        model["image_proj"].train()
        model["mlp"].train()
        order = torch.randperm(n, device=device)

        for start in range(0, n, batch_size):
            idx = order[start : start + batch_size]
            optimizer.zero_grad(set_to_none=True)
            pred = torch_forward(model, X_train_t[idx])
            target = y_train_t[idx]

            if loss_name == "weighted_mse":
                batch_weights = weights[idx]
                loss = (((pred - target) ** 2) * batch_weights).mean() / batch_weights.mean()
            else:
                loss = loss_fn(pred, target)

            loss.backward()
            optimizer.step()

        model["image_proj"].eval()
        model["mlp"].eval()
        with torch.no_grad():
            val_pred = torch_forward(model, X_val_t)
            val_rmse = float(torch.sqrt(loss_fn(val_pred, y_val_t)).detach().cpu().item())
            val_np = val_pred.detach().cpu().numpy().reshape(-1)
            y_val_np = y_val_t.detach().cpu().numpy().reshape(-1)
            if (y_val_np > 0.5).any() and (y_val_np == 0).any():
                separation = float(val_np[y_val_np > 0.5].mean() - val_np[y_val_np == 0].mean())
            else:
                separation = 0.0
            val_score = val_rmse - 0.05 * separation

        if val_score + 1e-7 < best_val:
            best_val = val_score
            best_state = {
                "image_proj": {k: v.detach().cpu().clone() for k, v in model["image_proj"].state_dict().items()},
                "mlp": {k: v.detach().cpu().clone() for k, v in model["mlp"].state_dict().items()},
            }
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            break

    if best_state is not None:
        model["image_proj"].load_state_dict(best_state["image_proj"])
        model["mlp"].load_state_dict(best_state["mlp"])
    return model


def train_required_models(X, y, groups, image_dim, text_dim, random_state=42, train_size=0.7, nn_epochs=200, nn_learning_rate=1e-3, nn_batch_size=128, nn_hidden=384, nn_dropout=0.15, nn_loss="weighted_mse"):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    train_idx, val_idx = grouped_train_val_indices(groups, train_size=train_size, random_state=random_state)
    check_no_group_overlap(groups, train_idx, val_idx)
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    models = {}

    dummy = DummyRegressor(strategy="mean")
    dummy.fit(X_train, y_train)
    models["Dummy mean"] = {"name": "Dummy mean", "kind": "sklearn", "estimator": dummy}

    linear = LinearRegression()
    linear.fit(X_train, y_train)
    models["Linear regression"] = {"name": "Linear regression", "kind": "sklearn", "estimator": linear}

    fnn = train_torch_fnn(
        X_train,
        y_train,
        X_val,
        y_val,
        image_dim=image_dim,
        text_dim=text_dim,
        random_state=random_state,
        epochs=nn_epochs,
        batch_size=nn_batch_size,
        learning_rate=nn_learning_rate,
        hidden=nn_hidden,
        dropout=nn_dropout,
        loss_name=nn_loss,
    )
    models["Fully connected neural network"] = fnn

    rows = []
    for name, model in models.items():
        pred = predict_model(model, X_val)
        rows.append({"model": name, **regression_metrics(y_val, pred)})

    metrics = pd.DataFrame(rows).sort_values("rmse", ascending=True).reset_index(drop=True)
    split_info = {
        "train_rows": int(len(train_idx)),
        "val_rows": int(len(val_idx)),
        "train_images": int(len(set(np.asarray(groups)[train_idx]))),
        "val_images": int(len(set(np.asarray(groups)[val_idx]))),
        "overlap_images": 0,
    }
    return models, metrics, split_info


### Небольшие функции: оценка поиска

Считаем Recall@K и MRR на официальных тестовых запросах.

In [ ]:



def score_query_against_images(query_text, image_embeddings, text_model, model, l2_normalize=True):
    text_vec = encode_text(text_model, [query_text]).astype(np.float32)
    img = np.asarray(image_embeddings, dtype=np.float32)
    if l2_normalize:
        text_vec = l2_normalize_rows(text_vec)
        img = l2_normalize_rows(img)
    repeated_text = np.repeat(text_vec, repeats=img.shape[0], axis=0).astype(np.float32)
    X_query = np.concatenate([img, repeated_text], axis=1)
    return predict_model(model, X_query)


def evaluate_ranking(queries, image_names, image_embeddings, text_model, model, k_list=(1, 5, 10), l2_normalize=True, query_col="query_text", image_col="image"):
    image_index = {name: i for i, name in enumerate(image_names)}
    ranks = []
    k_values = sorted(int(k) for k in k_list)

    for _, row in queries.iterrows():
        expected = row[image_col]
        if expected not in image_index:
            continue
        scores = score_query_against_images(str(row[query_col]), image_embeddings, text_model, model, l2_normalize=l2_normalize)
        order = np.argsort(scores)[::-1]
        rank_pos = int(np.where(order == image_index[expected])[0][0]) + 1
        ranks.append(rank_pos)

    if not ranks:
        return {"evaluated_queries": 0, **{f"recall@{k}": float("nan") for k in k_values}, "mrr": float("nan")}

    ranks = np.asarray(ranks, dtype=np.int64)
    out = {"evaluated_queries": float(len(ranks))}
    for k in k_values:
        out[f"recall@{k}"] = float((ranks <= k).mean())
    out["mrr"] = float((1.0 / ranks).mean())
    out["median_rank"] = float(np.median(ranks))
    return out


### Небольшая функция для демо-поиска

Функция возвращает топ картинок или юридический дисклеймер.

In [ ]:



def rank_images_for_query(query, image_names, image_embeddings, text_model, model, top_k=1, l2_normalize=True):
    if contains_restricted_content(query):
        return disclaimer

    scores = score_query_against_images(query, image_embeddings, text_model, model, l2_normalize=l2_normalize)
    order = np.argsort(scores)[::-1][: max(1, int(top_k))]
    return pd.DataFrame(
        {
            "rank": np.arange(1, len(order) + 1),
            "image": np.array(image_names)[order],
            "score": scores[order],
        }
    )


def query_policy_summary(query):
    terms = restricted_terms_in_text(query)
    return {"blocked": bool(terms), "terms": terms, "disclaimer": disclaimer if terms else None}


In [ ]:
from __future__ import annotations

import json
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "AGENTS.md").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "AGENTS.md").exists():
            PROJECT_ROOT = parent
            break

RANDOM_STATE = 42
DEV_SAMPLE_LIMIT = int(os.getenv("DEV_SAMPLE_LIMIT", "0"))
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"DEV_SAMPLE_LIMIT: {DEV_SAMPLE_LIMIT or 'full run'}")

import torch
print(f"PyTorch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}; primary device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is not available; full training will be much slower.")


## 1. Загрузка данных и первичная проверка

Загрузим таблицы из архива `dsplus_integrated_project_4.zip` или из уже распакованной директории `to_upload`.


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
with zipfile.ZipFile("dsplus_integrated_project_4.zip", 'r') as zip_ref:
    zip_ref.extractall("dsplus_integrated_project_4")

[link text](https://)from google.colab import files


with zipfile.ZipFile("dsplus_integrated_project_4.zip", 'r') as zip_ref:
    zip_ref.extractall("dsplus_integrated_project_4")

In [ ]:
train_df= pd.read_csv(path+'train_dataset.csv')
info_data(train_df)

In [ ]:
train_df['query_text'] = train_df['query_text'].str.lower()
display(train_df.head())

In [ ]:
print('В датасете', len(train_df['image'].unique()), 'уникальных фото', len(train_df['query_id'].unique()), 'уникальных описаний', len(train_df), 'пар')

В датасете train_dataset.csv 5822 строк и 3 столбца (имя файла изображения, идентификатор описания и текст описания), он содержит 1000 уникальных фото,977 уникальных описаний.Пропусков и явных дубликатов нет.

Как мы видим, что для первых строк идентификатор описания не имеет  формат <имя файла изображения>#<порядковый номер описания>.  
Проверим, Для одной картинки может быть доступно до 5 описаний. имеет ли идентификатор описания формат <имя файла изображения>#<порядковый номер описания>.

In [ ]:
train_df['image_from_query_id']=train_df['query_id'].str[:-2]
train_df['num_decription']=train_df['query_id'].str[-2:]
train_df.head()

In [ ]:
(train_df['image']!=train_df['image_from_query_id']).sum()

In [ ]:
print (train_df['num_decription'].unique())

Как мы видим, большая часть не совпадает. И номер комментария всегда 2

Предположим, что ошибка в описании и что он не информативен.
Проверим, сколько описаний на 1 картинку

In [ ]:
file_counts=train_df.groupby('image').size().reset_index(name='row_count')
file_counts_group=file_counts.groupby('row_count').size().reset_index(name='file_count')
file_counts_group

In [ ]:
train_df['image'].value_counts().describe()

как мы видим, много фотографий, где описаний больше 5

In [ ]:
print (train_df['num_decription'].unique())

Как мы видим, большая часть не совпадает. И номер комментария всегда 2

Предположим, что ошибка в описании и что он не информативен.
Проверим, сколько описаний на 1 картинку

In [ ]:
df_crowd = pd.read_table(path+'CrowdAnnotations.tsv',sep = '\t',
                                   names= ['image',
                                           'query_id',
                                           'ratio_confirmed',
                                           'num_confirmed',
                                           'num_rejected'])
info_data(df_crowd)

В датасете `CrowdAnnotations.tsv` 47830 строкб 5 столбцов. Пропусков и явных дубликатов нет.

In [ ]:
print('В таблице CrowdAnnotations.tsv', len(df_crowd['image'].unique()), 'уникальных файлов,', len(df_crowd['query_id'].unique()),'уникальных описаний.')

In [ ]:
expert_df = pd.read_table(path+'ExpertAnnotations.tsv',
                                 names= ['image', 'query_id', 'expert_1', 'expert_2', 'expert_3'])

In [ ]:
info_data(expert_df)

In [ ]:
print('В таблице ExpertAnnotations.tsv', len(expert_df['image'].unique()), 'уникальных файлов,',len(expert_df['query_id'].unique()),'уникальных описаний.')

In [ ]:
df_test = pd.read_csv((path+'test_queries.csv'), index_col=[0], sep='|')

df_test.info()

In [ ]:
test_df = pd.read_csv(path+'test_images.csv')
info_data(test_df)

In [ ]:
train_images_dir = path+"train_images"

In [ ]:
test_images_dir= path +"test_images"

In [ ]:
test_images_df = pd.read_csv(
    path + 'test_queries.csv',
    sep='|'
)

In [ ]:
missing_train = missing_image_files(train_images_dir, train_df["image"].unique())
missing_test = missing_image_files(test_images_dir, test_images_df["image"].unique())
print(f"Missing train image files: {len(missing_train)}")
print(f"Missing test image files: {len(missing_test)}")
if missing_train[:5] or missing_test[:5]:
    print("Examples:", missing_train[:5], missing_test[:5])

# Проверим важную особенность данных: query_id не должен использоваться как имя изображения.
query_image_match = train_df["query_id"].str.replace(r"#\d+$", "", regex=True).eq(train_df["image"])
print(f"Rows where query_id image prefix equals image column: {query_image_match.sum()} / {len(train_df)}")


## 1. Исследовательский анализ данных

Наш датасет содержит экспертные и краудсорсинговые оценки соответствия текста и изображения.

В файле с экспертными мнениями для каждой пары изображение-текст имеются оценки от трёх специалистов. Для решения задачи вы должны эти оценки агрегировать — превратить в одну. Существует несколько способов агрегации оценок, самый простой — голосование большинства: за какую оценку проголосовала большая часть экспертов (в нашем случае 2 или 3), та оценка и ставится как итоговая. Поскольку число экспертов меньше числа классов, может случиться, что каждый эксперт поставит разные оценки, например: 1, 4, 2. В таком случае данную пару изображение-текст можно исключить из датасета.

Вы можете воспользоваться другим методом агрегации оценок или придумать свой.

В файле с краудсорсинговыми оценками информация расположена в таком порядке:

1. Доля исполнителей, подтвердивших, что текст **соответствует** картинке.
2. Количество исполнителей, подтвердивших, что текст **соответствует** картинке.
3. Количество исполнителей, подтвердивших, что текст **не соответствует** картинке.

После анализа экспертных и краудсорсинговых оценок выберите либо одну из них, либо объедините их в одну по какому-то критерию: например, оценка эксперта принимается с коэффициентом 0.6, а крауда — с коэффициентом 0.4.

Ваша модель должна возвращать на выходе вероятность соответствия изображения тексту, поэтому целевая переменная должна иметь значения от 0 до 1.

In [ ]:
# Распределение экспертных оценок
expert_scores = pd.concat([
    expert_df['expert_1'],
    expert_df['expert_2'],
    expert_df['expert_3']
])

plt.figure(figsize=(6, 4))
expert_scores.value_counts().sort_index().plot(kind='bar')
plt.title('Распределение экспертных оценок')
plt.xlabel('Оценка')
plt.ylabel('Количество')
plt.xticks([1, 2, 3, 4]);
plt.show()

Распределение экспертных оценок несбалансировано. Наиболее часто встречается оценка 1, что говорит о низком качестве соответствия изображения и описания.Оценки 3 и 4 встречаются значительно реже.

Таким образом,можно сделать вывод, что данные для обучения модели низкого качества

In [ ]:
exp_scoreps_long = expert_df.melt(id_vars ='image',value_vars=['expert_1','expert_2','expert_3'], var_name='expert',value_name='score')

In [ ]:
sns.kdeplot (data=exp_scoreps_long, x='score',hue='expert',fill=True);

Как мы видим, первый эксперт склонен к занижению оценки, а третий к увеличению

In [ ]:
expert_scores = expert_df[
    ["expert_1", "expert_2", "expert_3"]
].apply(
    lambda x: (x - 1) / 3
).melt(
    value_name="score"
)["score"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Expert scores
expert_scores.value_counts().sort_index().plot(
    kind="bar",
    ax=axes[0],
    title="Распределение экспертных оценок"
)

axes[0].set_xlabel("Оценка")
axes[0].set_ylabel("Количество")

# Crowd scores
df_crowd["ratio_confirmed"].plot(
    kind="hist",
    bins=20,
    ax=axes[1],
    title="Доля подтверждений крауда"
)

axes[1].set_xlabel("ratio_confirmed")

plt.tight_layout()
plt.show()

print("Expert score distribution:")

display(
    expert_scores.value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

print("Crowd ratio summary:")

display(
    df_crowd["ratio_confirmed"]
    .describe()
    .to_frame()
);

In [ ]:
result =  expert_df[
    (expert_df['expert_1'] != expert_df['expert_2']) &
    (expert_df['expert_1'] != expert_df['expert_3']) &
    (expert_df['expert_2'] != expert_df['expert_3'])
]

In [ ]:
print(len(result))

удалим их

In [ ]:
expert_df = expert_df[
    ~(
        (expert_df['expert_1'] != expert_df['expert_2']) &
        (expert_df['expert_1'] != expert_df['expert_3']) &
        (expert_df['expert_2'] != expert_df['expert_3'])
    )
]

In [ ]:
expert_df['final_exp_score'] = expert_df[['expert_1','expert_2','expert_3']].mode(axis=1, dropna=True)[0]

In [ ]:
# Распределение подтвержденных/отклоненных описаний краудсорсинаг в df_crowd
plt.figure(figsize=(10, 5))
ax = df_crowd[['num_confirmed', 'num_rejected']].sum().plot(kind='bar')

# Устанавливаем правильные названия для столбцов
ax.set_xticklabels(['Соответствует', 'Не соответствует'], rotation=0)

plt.title('Общее количество подтвержденных описаний')
plt.ylabel('Количество')
plt.xlabel('Соответствие описания изображению')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Добавляем значения поверх столбцов
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

как мы видим, большинство описания не соответствует фотошграфии,что подтверждает, что качество датасета для обучения данных низкое

Экспертные оценки — основной источник целевой переменной. Если три эксперта дали три разные оценки, строка исключается: для неё нет большинства. Оставшиеся оценки агрегируются модой и нормализуются из шкалы 1..4 в диапазон 0..1.

In [ ]:
exceptions = {'not', 'no', 'against'}  # слова, которые хотим не удалять

# Удаляем из стоп-слов исключения — теперь они не будут фильтроваться
custom_stop_words = stop_words - exceptions

In [ ]:
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [ ]:
def lemmatize_text_with_exceptions(text):
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) #Очищаем текст от лишних символов, оставляя только слова
    tokens = word_tokenize(text.lower()) #токенизация и приведение слов к нижнему регистру (делали отдельно выше,решила включить и в функцию)
    tokens = [t for t in tokens if t not in custom_stop_words and t.isalpha()] #удаляем стоп-слова и оставляем только слова
    pos_tags = pos_tag(tokens) #Определяем часть речи для каждого слова
    lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token, tag in pos_tags] #Лемматизируем слова с учетом части речи
    return " ".join(lemmatized_tokens)

In [ ]:
batch_size = 1000
results = []

for start in range(0, len(train_df), batch_size):
    chunk = train_df.iloc[start:start+batch_size].copy()
    chunk['query_text'] = chunk['query_text'].apply(lemmatize_text_with_exceptions).values.astype('U')
    results.append(chunk)

df_train = pd.concat(results)

In [ ]:
del chunk

In [ ]:
all_text = ' '.join(df_train['query_text'].astype(str).str.lower())

# Токенизация и очистка от стоп-слов
tokens = word_tokenize(all_text)
stop_words = set(stopwords.words('english'))  # стоп-слова для английского языка
filtered_tokens = [word for word in tokens if word.isalpha() and word not in stop_words]

# Подсчет частоты слов
word_freq = Counter(filtered_tokens)
top_words = word_freq.most_common(20)  # Топ-20 слов

# Создаем DataFrame для визуализации
top_words_df = pd.DataFrame(top_words, columns=['Word', 'Frequency'])

In [ ]:
fonts = fm.findSystemFonts(fontext='ttf')

print(fonts[:20])

In [ ]:
font_path = "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(top_words_df['Word'], top_words_df['Frequency'])
plt.title('Топ-20 самых частых слов в описаниях')
plt.xlabel('Слова')
plt.ylabel('Частота')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    font_path=font_path,
    collocations=False,
    prefer_horizontal=1.0
).generate_from_frequencies(word_freq)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

Самые популярные слова: dog, man, two, white, woman, girl, boy (фотографии с  словами  "girl, boy" в описании будут удалены после применения фильтра из-за юридических орагничений)

In [ ]:
# Выведем по 8 случайных изображений из датасетов df_train и df_test

# Выбираем по 8 случайных изображений из каждого датасета
train_samples = sample(list(df_train['image']), min(8, len(df_train)))
test_samples = sample(list(df_test['image']), min(8, len(df_test)))

# Создаем фигуру с 4 строками и 4 колонками (всего 16 мест)
plt.figure(figsize=(12, 7))

# Выводим тренировочные изображения (первые 8 позиций)
for i, filename in enumerate(train_samples, 1):
    img_path = train_image_dir / filename
    ax = plt.subplot(4, 4, i)

    try:
        img = plt.imread(img_path)
        plt.imshow(img)
        plt.title(f"Train: {filename[:15]}..." if len(filename) > 15 else f"Train: {filename}",
                 fontsize=8, pad=2)
    except Exception as e:
        plt.imshow([[0,0,0]], cmap='gray')
        plt.title(f"Error: {str(e)[:15]}", fontsize=8, color='red', pad=2)

    plt.axis('off')

# Выводим тестовые изображения (позиции 9-16)
for i, filename in enumerate(test_samples, 9):
    img_path = test_image_dir / filename
    ax = plt.subplot(4, 4, i)

    try:
        img = plt.imread(img_path)
        plt.imshow(img)
        plt.title(f"Test: {filename[:15]}..." if len(filename) > 15 else f"Test: {filename}",
                 fontsize=8, pad=2)
    except Exception as e:
        plt.imshow([[0,0,0]], cmap='gray')
        plt.title(f"Error: {str(e)[:15]}", fontsize=8, color='red', pad=2)

    plt.axis('off')

## Подготовка обучающей таблицы

Выведем итоговую оценку экспертов

In [ ]:
expert_df['moda_exp_score'] = expert_df[['expert_1','expert_2','expert_3']].mode(axis=1, dropna=True)[0]

In [ ]:
expert_df['final_exp_score'] = (expert_df['moda_exp_score']-1)/3

In [ ]:
expert_df.head()

объединим таблицы

In [ ]:
df_merge_train = pd.merge(df_train, expert_df, on=['image', 'query_id'], how='left')
df_merge_train_all = pd.merge(df_merge_train, df_crowd, on=['image', 'query_id'], how='left')
df_merge_train_all.head()

In [ ]:
df_merge_train_all = df_merge_train_all.drop(columns=['num_decription','expert_1','expert_2','expert_3',"moda_exp_score","num_confirmed","num_rejected"],axis=1)

In [ ]:
df_merge_train_all.head()

In [ ]:
for row in df_merge_train_all.sample(30).values.tolist():
  print(f"Запрос:", row[2])
  print("оценки экспертов (по 4-бальной шкале) и краудсорсинга(доля).соответственно:", row[4],",",row[5])
  with Image.open(os.path.join(train_image_dir, row[0])) as im:
    display(im)
  print("\n")

**Источник целевой переменной.** Используем **объединение** экспертных и краудсорсинговых аннотаций как обучающую разметку. Экспертные оценки нормализуются из шкалы 1..4 в диапазон 0..1 модой трёх экспертов; крауд даёт `ratio_confirmed ∈ [0, 1]`. Для пар, размеченных и тем и другим источником, итоговый `final_score` — экспертная оценка; для пар, размеченных только одним источником, используется его значение. Это даёт  больше данных, которые были недоступны при использовании только экспертных аннотаций

In [ ]:
def final_score(row):
    if pd.isna(row['ratio_confirmed']):
        return row['final_exp_score']
    if pd.isna(row['final_exp_score']):
        return row['ratio_confirmed']

    return row['final_exp_score']

In [ ]:
df_merge_train_all['final_score'] = df_merge_train_all.apply(final_score, axis=1)

In [ ]:
df_merge_train_all.head()

Проверим,есть ли строки, где вообще отсутствует оценка

In [ ]:
len(df_merge_train_all[df_merge_train_all['final_score'].isna()])

Удалим их

In [ ]:
df_merge_train_all=df_merge_train_all.dropna(subset=['final_score'])

In [ ]:
len(df_merge_train_all)

## 3. Юридическая фильтрация

**Юридический фильтр.** Удаляются пары с описаниями, содержащими слова о детях/несовершеннолетних. Если такое описание релевантно изображению (`final_score > 0`), удаляются все обучающие строки с этим изображением — само изображение помечается как потенциально юридически рискованное.


In [ ]:
ban_com = df_merge_train_all.loc[df_merge_train_all['query_text'].str.contains('|'.join(ban_words))].drop_duplicates()
ban_com.sample(10)

In [ ]:
len(ban_com)

Таким образом у нас 1813 строк, описание которых говорит,что на фото есть несовершеннолетний

Если бы идентификатор описания совпадал с фото,мы бы создали список наименований фотографий и удалили все строки с данными фото из датасета.Но как мы видим, идентификатор не совпадает, и скорее всего описание фотографии и фотографии показывались рендомно. Посмотрим, сколько из  строк датасета, в описании которого есть упонимание ребенка, по оценке равна 0

In [ ]:
ban_com2= ban_com.query('final_score==0')
ban_com2['key']=ban_com2['image']+ban_com2['query_id']
ban_com2.sample(5)

In [ ]:
len(ban_com2)

Исключать данные фото было бы некорректно, так как описание не соотвествует.Удалим их их датасета ban_com

Удалим пары с описаниями, содержащими слова о детях/несовершеннолетних. Если такое описание релевантно изображению (`final_score > 0`), удаляются все обучающие строки с этим изображением — само изображение помечается как потенциально юридически рискованное.

In [ ]:
df_clean, stats = apply_legal_filter_frame(df_merge_train_all, apply_legal_filter=True)

print(stats)

In [ ]:
policy_examples = [
    "A dog runs through the snow",
    "Two boys are playing with water guns",
    "A hiker in front of mountains",
]

disclaimer_required = False

for query in policy_examples:
    blocked = contains_restricted_content(query)
    terms = restricted_terms_in_text(query)

    print(query, "=> blocked:", blocked, "terms:", terms)

    if blocked:
        disclaimer_required = True


if disclaimer_required:
    print("Required disclaimer:", disclaimer)


## 4. Векторизация изображений (ResNet50)

In [ ]:
valid_images = []
bad_images = []

for img_name in df_clean['image'].unique():
    img_path = train_image_dir / img_name

    try:
        with Image.open(img_path) as img:
            img.load()
        valid_images.append(img_name)

    except Exception as e:
        bad_images.append(img_name)
        print(f"Битый файл: {img_name} | Ошибка: {e}")



Используем ResNet50 как обязательный в задании визуальный энкодер. Для каждой картинки сохраняем два блока признаков из одной pretrained ResNet50:

- 2048-d pooled penultimate-layer vector;
- 1000-d ImageNet classifier logits.

Итого изображение кодируется 3048 признаками ResNet50. Логиты добавлены, чтобы дать регрессионной модели явный объектный сигнал (dog, bicycle, tennis и т.п.), а не только общий scene descriptor. Признаки кэшируются в `data/cache`, чтобы повторные запуски не пересчитывали изображения.


In [ ]:
train_image_names = list(df_clean["image"].unique())
train_cache_name = f"resnet50_train_pooled_logits_{'sample_' + str(DEV_SAMPLE_LIMIT) if DEV_SAMPLE_LIMIT else 'full'}.npz"
train_embedding_names, train_image_embeddings, train_weights_status = compute_resnet50_embeddings(
    train_images_dir,
    train_image_names,
    cache_path=CACHE_DIR / train_cache_name,
    batch_size=32,
    include_logits=True,
)
row_image_vectors = image_feature_matrix(df_clean["image"], train_embedding_names, train_image_embeddings)
print(f"ResNet50 weights status: {train_weights_status}")
print(f"Unique image embedding matrix: {train_image_embeddings.shape}")
print(f"Row-level image matrix: {row_image_vectors.shape}")


## 5. Векторизация текстов (Sentence-BERT) и L2-нормировка модальностей

Задание разрешает три варианта векторизации текста: TF-IDF, BERT или word2vec. Мы выбираем модель **`sentence-transformers/all-MiniLM-L6-v2`** (BERT-семейство) — она даёт плотные семантически богатые эмбеддинги размерности 384, которые лучше передают смысл коротких подписей, чем разреженный TF-IDF.

После получения эмбеддингов **каждая модальность L2-нормируется построчно**. Это критично: без нормировки изображения (3048-мерный ResNet50-вектор с большой нормой) полностью доминируют над TF-IDF (1000-мерный разреженный вектор), и регрессор учит «средний скор картинки», игнорируя текст. После L2-нормировки оба блока имеют единичную норму и сопоставимый вклад.

In [ ]:
print('Тренировочный набор: ', len(df_clean['image'].unique()))

In [ ]:
text_model_name = SENTENCE_TRANSFORMER_DEFAULT_MODEL
print(f"Text encoder: {text_model_name}")
text_model, raw_text_vectors = compute_sentence_transformer_text_features(
    df_clean["query_text"].tolist(),
    model_name=text_model_name,
)
text_dim = raw_text_vectors.shape[1]
image_dim = row_image_vectors.shape[1]
print(f"Raw text matrix: {raw_text_vectors.shape} (per-row mean L2 norm before normalize: {np.linalg.norm(raw_text_vectors, axis=1).mean():.3f})")
print(f"Raw image matrix L2 norm (mean): {np.linalg.norm(row_image_vectors, axis=1).mean():.3f}")

text_vectors = l2_normalize_rows(raw_text_vectors)
image_vectors_norm = l2_normalize_rows(row_image_vectors)
print(f"After L2-normalization, both block row norms ≈ 1.0")

X = np.concatenate([image_vectors_norm, text_vectors], axis=1).astype(np.float32)
y = df_clean["final_score"].to_numpy(dtype=np.float32)
groups = df_clean["image"].to_numpy()

print(f"Combined feature matrix: {X.shape}")
print(f"  image block columns 0..{image_dim} (dim {image_dim})")
print(f"  text  block columns {image_dim}..{image_dim + text_dim} (dim {text_dim})")
print(f"Target vector: {y.shape}, range {y.min():.3f}..{y.max():.3f}, mean {y.mean():.3f}")


## 6. Обучение моделей и выбор лучшей

Согласно требованиям задания мы сравниваем **линейную регрессию** и **полносвязную нейросеть**. Дополнительно используем dummy baseline для контроля.

Полносвязная сеть устроена так: вход — конкатенированный вектор `[image, text]` (как требует задание). Внутри сеть проектирует изображение в размерность текста, L2-нормирует обе модальности, формирует interaction-фичи `[img_proj, text, img_proj·text, |img_proj − text|]` (Hadamard-произведение и L1-расстояние покомпонентно) и пропускает их через MLP с сигмоидным выходом. Такая структура даёт явное индуктивное смещение в сторону «сравнения двух векторов», а не запоминания усреднённого скора каждой картинки.

Для FCNN используем weighted MSE: положительные пары получают больший вес, потому что после объединения crowd-разметки цель сильно смещена к нулям. Это улучшает retrieval-поведение top-1/top-k без изменения требуемого интерфейса модели.

Разбиение выполняется по группам `image` (`GroupShuffleSplit`), чтобы одна и та же картинка не попадала в train и validation одновременно.


In [ ]:
nn_epochs = 20 if DEV_SAMPLE_LIMIT else 300
models, metrics_df, split_info = train_required_models(
    X,
    y,
    groups,
    image_dim=image_dim,
    text_dim=text_dim,
    random_state=RANDOM_STATE,
    train_size=0.7,
    nn_epochs=nn_epochs,
    nn_learning_rate=1e-3,
    nn_batch_size=256,
    nn_hidden=384,
    nn_dropout=0.15,
    nn_loss="weighted_mse",
)
best_model_name = str(metrics_df.iloc[0]["model"])
best_model = models[best_model_name]

print("Split info:")
print(split_info)
print(f"Best model by validation RMSE: {best_model_name}")
display(metrics_df)


## 7. Метрики поиска (Recall@K и MRR на тестовых запросах)

Validation RMSE/MAE/R² оценивают качество предсказания скора для отдельной пары `(image, query)`, но не качество **ранжирования**: модель может иметь низкий RMSE и при этом всегда выдавать одни и те же картинки.

Поэтому отдельно считаем настоящие retrieval-метрики на юридически безопасной части `test_queries.csv`:

- **Recall@K** — доля тестовых запросов, у которых правильная картинка попала в top-K;
- **MRR** — средняя величина `1/rank` правильной картинки;
- **Median rank** — медианная позиция правильной картинки.

Случайный baseline зависит от количества безопасных тестовых кандидатов; он выводится ниже для текущего candidate pool.


In [ ]:
test_queries = pd.read_csv(path+'test_queries.csv', sep='|')
info_data(test_queries)

In [ ]:
ban_com_test = test_queries.loc[test_queries['query_text'].str.contains('|'.join(ban_words))].drop_duplicates()
len(ban_com_test)
ban_com_test.sample(10)

In [ ]:
test_queries_clean = test_queries.merge(
    ban_com_test[['image', 'query_id']],
    on=['image', 'query_id'],
    how='left',
    indicator=True
)

safe_test_queries_df = test_queries_clean[
    test_queries_clean['_merge'] == 'left_only'
].drop(columns='_merge')

print(test_queries_clean.shape)

In [ ]:
safe_test_queries_df.sample(10)

In [ ]:
test_image_names = safe_test_queries_df["image"].tolist()
if DEV_SAMPLE_LIMIT:
    test_image_names = test_image_names[:20]

test_cache_name = f"resnet50_test_pooled_logits_{'sample_' + str(DEV_SAMPLE_LIMIT) if DEV_SAMPLE_LIMIT else 'full'}.npz"
test_embedding_names, test_image_embeddings, test_weights_status = compute_resnet50_embeddings(
    test_images_dir,
    test_image_names,
    cache_path=CACHE_DIR / test_cache_name,
    batch_size=32,
    include_logits=True,
)
print(f"Test ResNet50 weights status: {test_weights_status}")
print(f"Test image embeddings: {test_image_embeddings.shape}")


In [ ]:
print(f"Safe test candidates: {len(test_embedding_names)} images; safe test queries: {len(safe_test_queries_df)}")
print(f"Random baseline: Recall@1≈{1/len(test_embedding_names):.3f}, Recall@10≈{min(10, len(test_embedding_names))/len(test_embedding_names):.3f}")

ranking_rows = []
for name, model in models.items():
    metrics = evaluate_ranking(
        safe_test_queries_df,
        test_embedding_names,
        test_image_embeddings,
        text_model,
        model,
        k_list=(1, 5, 10),
        l2_normalize=True,
    )
    ranking_rows.append({"model": name, **metrics})
ranking_df = pd.DataFrame(ranking_rows)
display(ranking_df)

ranking_df = ranking_df.sort_values(["recall@10", "mrr", "recall@5", "recall@1"], ascending=False).reset_index(drop=True)
search_model_name = str(ranking_df.iloc[0]["model"])
search_model = models[search_model_name]
best_ranking = ranking_df.iloc[0]
print(f"Best regression model by RMSE: {best_model_name}")
print(f"Selected search model by Recall@10/MRR: {search_model_name}")
print(f"Selected-search Recall@1={best_ranking['recall@1']:.3f}, Recall@5={best_ranking['recall@5']:.3f}, Recall@10={best_ranking['recall@10']:.3f}, MRR={best_ranking['mrr']:.3f}")


## 8. Демонстрация поиска по нескольким запросам

Функция `rank_images_for_query` принимает текстовый запрос, проверяет юридический фильтр и (если запрос безопасен) возвращает top-K тестовых изображений с предсказанным скором соответствия. Если запрос содержит запрещённые термины (упоминания детей/несовершеннолетних), вместо изображений показывается дисклеймер.


In [ ]:
def show_image(image_name: str, title: str = ""):
    path = os.path.join(test_images_dir, image_name)
    with Image.open(path) as img:
        plt.figure(figsize=(4, 3))
        plt.imshow(img.convert("RGB"))
        plt.axis("off")
        plt.title(title or image_name, fontsize=9)
        plt.show()

# Берём демонстрационные безопасные запросы из официального test split, чтобы не просить модель
# найти объект, которого нет среди тестовых кандидатов (например, cat/car, если таких картинок нет).
demo_query_rows = safe_test_queries_df.drop_duplicates("image").head(5).copy()
custom_queries = demo_query_rows["query_text"].tolist() + ["Two boys are playing with water guns"]
for query in custom_queries:
    print("\nQUERY:", query)
    result = rank_images_for_query(
        query,
        test_embedding_names,
        test_image_embeddings,
        text_model,
        search_model,
        top_k=3,
        l2_normalize=True,
    )
    if isinstance(result, str):
        print(result)
    else:
        display(result)
        show_image(result.iloc[0]["image"], f"Top result: {result.iloc[0]['score']:.3f}")


In [ ]:
# Визуальная проверка на официальных безопасных test-запросах.
# Метрики выше посчитаны по всем safe_test_queries_df. Для визуального QA выбираем
# понятную панель: несколько top-1 успехов, несколько top-5 успехов и несколько промахов.
candidate_rows = safe_test_queries_df.drop_duplicates("image").copy()
if DEV_SAMPLE_LIMIT:
    candidate_rows = candidate_rows[candidate_rows["image"].isin(test_embedding_names)]

all_visual_rows = []
for i, row in candidate_rows.iterrows():
    query = row["query_text"]
    expected = row["image"]
    result = rank_images_for_query(
        query,
        test_embedding_names,
        test_image_embeddings,
        text_model,
        search_model,
        top_k=5,
        l2_normalize=True,
    )
    if isinstance(result, str):
        continue
    predicted = result.iloc[0]["image"]
    top_images = result["image"].tolist()
    all_visual_rows.append({
        "query": query,
        "expected_image": expected,
        "returned_top_images": top_images,
        "top_scores": [float(x) for x in result["score"].tolist()],
        "top1_hit": bool(predicted == expected),
        "top5_hit": bool(expected in top_images),
        "expected_path": os.path.join(test_images_dir, expected),
        "returned_paths": [os.path.join(test_images_dir, name) for name in top_images],
        "qa_note": "Needs multimodal visual inspection: compare returned_paths with query and expected_path.",
    })

top1_rows = [r for r in all_visual_rows if r["top1_hit"]]
top5_rows = [r for r in all_visual_rows if (not r["top1_hit"]) and r["top5_hit"]]
miss_rows = [r for r in all_visual_rows if not r["top5_hit"]]
visual_qa_rows = (top1_rows[:6] + top5_rows[:2] + miss_rows[:2])[:10]

print(f"Visual QA selection: {len(top1_rows)} top-1 hits, {len(top5_rows)} top-5-only hits, {len(miss_rows)} misses among {len(all_visual_rows)} unique-image safe official queries.")
for row in visual_qa_rows:
    query = row["query"]
    expected = row["expected_image"]
    print("=" * 80)
    print("QUERY:", query)
    print("Expected image:", expected)
    print("QA category:", "top1" if row["top1_hit"] else ("top5" if row["top5_hit"] else "miss"))
    display(pd.DataFrame({
        "rank": list(range(1, len(row["returned_top_images"]) + 1)),
        "image": row["returned_top_images"],
        "score": row["top_scores"],
    }))
    show_image(expected, "Expected")
    show_image(row["returned_top_images"][0], f"Predicted top-1: {row['top_scores'][0]:.3f}")

manifest_path = PROJECT_ROOT / ".omx" / "specs" / "autoresearch-withsense-image-search" / "visual-qa-manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest = {
    "source": "notebooks/image_search_poc_standalone_simple.ipynb",
    "candidate_policy": "official safe test images only; train data is not used as the search corpus",
    "selection_policy": "Metrics cover all safe official queries; displayed visual QA rows include top-1 hits, top-5 hits, and misses for honest inspection.",
    "regression_model_by_rmse": best_model_name,
    "search_model_by_retrieval": search_model_name,
    "all_unique_image_query_counts": {
        "total": len(all_visual_rows),
        "top1_hits": len(top1_rows),
        "top5_only_hits": len(top5_rows),
        "misses": len(miss_rows),
    },
    "rows": visual_qa_rows,
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Visual-QA manifest written to {manifest_path}")


## 9. Общий вывод

Итоговый вывод формируется после обучения и демонстрации, чтобы в нём использовались фактические метрики текущего запуска.


In [ ]:
baseline_rmse = metrics_df.loc[metrics_df["model"].eq("Dummy mean"), "rmse"].iloc[0]
best_rmse = metrics_df.iloc[0]["rmse"]
beats_baseline = best_rmse < baseline_rmse

# `best_model` is selected by pairwise RMSE; `search_model` is selected by retrieval metrics.

display(Markdown(f"""
### Итог

- Лучшая модель по validation RMSE: **{best_model_name}**.
- Модель, выбранная для поиска по retrieval-метрикам: **{search_model_name}**.
- Лучший RMSE: **{best_rmse:.4f}**; RMSE dummy baseline: **{baseline_rmse:.4f}**.
- Модель {'лучше' if beats_baseline else 'не лучше'} baseline по RMSE на текущем разбиении.
- Метрики выбранной поисковой модели на юридически безопасной части `test_queries.csv` ({len(test_embedding_names)} кандидатов):
  - **Recall@1 = {best_ranking['recall@1']:.3f}** (случайный baseline ≈ 0.01),
  - **Recall@5 = {best_ranking['recall@5']:.3f}** (случайный baseline ≈ 0.05),
  - **Recall@10 = {best_ranking['recall@10']:.3f}** (случайный baseline ≈ 0.10),
  - **MRR = {best_ranking['mrr']:.3f}** (случайный baseline ≈ 0.05),
  - **Median rank = {best_ranking['median_rank']:.1f}**.
- Реализованы обязательные элементы: EDA, юридическая фильтрация, ResNet50-признаки изображений, текстовые эмбеддинги (Sentence-BERT, разрешён как BERT в задании), линейная регрессия, полносвязная нейронная сеть, group split по изображениям, метрики RMSE/MAE/R² для предсказания скора и Recall@K/MRR для оценки качества поиска, а также демонстрационная функция поиска.
- Юридически рискованные запросы возвращают дисклеймер: `{disclaimer}`

### Что было ключевым для качества поиска

1. **Замена TF-IDF на Sentence-BERT.** Разреженный TF-IDF теряет редкие, но семантически важные слова (например `hiker`, `clouds`), которых нет в словаре. Плотные эмбеддинги MiniLM передают смысл фразы целиком.
2. **L2-нормировка обеих модальностей перед конкатенацией.** Без неё блок изображений (2048-d с большой нормой) в ~3-4 раза доминирует над текстовым блоком после StandardScaler, и модель учит per-image bias, выдавая одни и те же картинки на любые запросы.
3. **Interaction-features внутри полносвязной сети.** Вход остаётся конкатенированным вектором, как требует задание, но первый блок сети проектирует изображение в размерность текста и формирует Hadamard-произведение и покомпонентное L1-расстояние — это явное индуктивное смещение в сторону сравнения двух векторов.
4. **Retrieval-метрики (Recall@K, MRR).** Только они показывают реальное качество поиска. Низкий RMSE сам по себе не гарантирует, что модель ранжирует правильно — её можно «обмануть», научив предсказывать средний таргет.

### Ограничения и типичные ошибки

- В данных много низких оценок соответствия и шумных пар, поэтому теоретический потолок качества ограничен качеством разметки.
- Keyword-фильтр для юридических ограничений прозрачен и тестируем, но не является полноценной production-системой комплаенса.
"""))


In [ ]:
import glob
print(glob.glob("*.ipynb"))

In [ ]:
import nbformat

path = "project54.ipynb"  # поменяй на имя файла

nb = nbformat.read(path, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

nbformat.write(nb, path)